In [1]:
import jax
import jax.numpy as jnp
from unscented import UnscentedKalmanFilter

In [2]:
#Testing unscented transform with toy example
def unscented_transform(mean, cov, f, alpha=1e-3, beta=2.0, kappa=0.0):
    d = mean.shape[0]
    lam = alpha**2 * (d + kappa) - d
    print("lambda:", lam)

    S = jnp.linalg.cholesky((d + lam) * cov) #Matrix square root
    print("S:", S)

    sigma_points = jnp.concatenate([
        mean[None, :], #changes the shape of mean from (2,) into (1, 2)
        mean[None, :] + S.T,
        mean[None, :] - S.T,
    ], axis=0)
    print("sigma_points:", sigma_points)

    Wm = jnp.concatenate([ #creates an array of W(m) value from 0 to 2d
        jnp.array([lam / (d + lam)]),
        jnp.full(2 * d, 1.0 / (2 * (d + lam)))
    ])
    print("Wm:", Wm)

    Wc = Wm.at[0].add(1 - alpha**2 + beta)
    print("Wc:", Wc)

    transformed = jnp.stack([f(sp) for sp in sigma_points])
    print("transformed:", transformed)

    mean_pred = jnp.sum(Wm[:, None] * transformed, axis=0)
    print("mean_pred:", mean_pred)

    diff = transformed - mean_pred
    cov_pred = jnp.einsum('i,ij,ik->jk', Wc, diff, diff)
    print("cov_pred:", cov_pred)

    return mean_pred, cov_pred

mean = jnp.array([0.0, 0.0])
cov = jnp.eye(2)
unscented_transform(mean, cov, lambda x: x)

lambda: -1.999998
S: [[0.00141421 0.        ]
 [0.         0.00141421]]
sigma_points: [[ 0.          0.        ]
 [ 0.00141421  0.        ]
 [ 0.          0.00141421]
 [-0.00141421  0.        ]
 [ 0.         -0.00141421]]
Wm: [-999999.  250000.  250000.  250000.  250000.]
Wc: [-999996.  250000.  250000.  250000.  250000.]
transformed: [[ 0.          0.        ]
 [ 0.00141421  0.        ]
 [ 0.          0.00141421]
 [-0.00141421  0.        ]
 [ 0.         -0.00141421]]
mean_pred: [0. 0.]
cov_pred: [[1. 0.]
 [0. 1.]]


(Array([0., 0.], dtype=float32),
 Array([[1., 0.],
        [0., 1.]], dtype=float32))

In [3]:
#linear sanity check - using a matrix with closed form solution
A = jnp.array([[2.0, 1.0], [0.0, 3.0]])

m_pred, c_pred = unscented_transform(mean, cov, lambda x: A @ x)

print("UKF mean:", m_pred)
print("Closed-form mean:", A @ mean)

print("UKF cov:", c_pred)
print("Closed-form cov:", A @ cov @ A.T)

print("Means match:", jnp.allclose(m_pred, A @ mean, atol=1e-4))
print("Covariances match:", jnp.allclose(c_pred, A @ cov @ A.T, atol=1e-4))

lambda: -1.999998
S: [[0.00141421 0.        ]
 [0.         0.00141421]]
sigma_points: [[ 0.          0.        ]
 [ 0.00141421  0.        ]
 [ 0.          0.00141421]
 [-0.00141421  0.        ]
 [ 0.         -0.00141421]]
Wm: [-999999.  250000.  250000.  250000.  250000.]
Wc: [-999996.  250000.  250000.  250000.  250000.]
transformed: [[ 0.          0.        ]
 [ 0.00282843  0.        ]
 [ 0.00141421  0.00424264]
 [-0.00282843  0.        ]
 [-0.00141421 -0.00424264]]
mean_pred: [0. 0.]
cov_pred: [[5.        3.0000002]
 [3.0000002 9.000001 ]]
UKF mean: [0. 0.]
Closed-form mean: [0. 0.]
UKF cov: [[5.        3.0000002]
 [3.0000002 9.000001 ]]
Closed-form cov: [[5. 3.]
 [3. 9.]]
Means match: True
Covariances match: True
